## Read in data

In [0]:
df = spark.read.csv("/Volumes/workspace/default/ecommerce_data/customer-behavior-and-churn-prediction-dataset/ecommerce_customer_data.csv", header="true", inferSchema="true")
display(df)

## Drop duplicates, inspect schema and nulls

In [0]:
# Inspect schema
df.printSchema()

In [0]:
# Display number of rows
df.count()

In [0]:
# Drop duplicate entries
df = df.dropDuplicates()
df.count()

In [0]:
# Count missing values in all columns
from pyspark.sql.functions import isnan, when, count, col
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

## EDA: Age, Gender, Annual_Income, Spending_Score, Membership_Years, Online_Purchases, Discount_Usage, Churn

In [0]:
# Drop Customer_ID
df = df.drop("Customer_ID")

# Convert Gender to Male
df = df.withColumn("Male", when(df.Gender == "Male", 1).otherwise(0))
df.show(5)

### Correlation matrices

In [0]:
# Examine df.dtypes
df.dtypes

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.mllib.stat import Statistics


# Prepare vector of df columns for correlation
numeric_cols = [col for col, dtype in df.dtypes if dtype in ['int', 'double', 'float', 'decimal']]
vector_col = "corr_features"

assembler = VectorAssembler(inputCols=numeric_cols, outputCol=vector_col)
df_vector = assembler.transform(df).select(vector_col)

# Pearson correlation matrix
pearson_matrix = Correlation.corr(df_vector, vector_col, "pearson").collect()[0][0]
pearson_matrix = pearson_matrix.toArray().tolist()
pearson_df = spark.createDataFrame(pearson_matrix, numeric_cols)
display(pearson_df)

# Spearman correlation matrix
spearman_matrix = Correlation.corr(df_vector, vector_col, "spearman").collect()[0][0]
spearman_matrix = spearman_matrix.toArray().tolist()
spearman_df = spark.createDataFrame(spearman_matrix, numeric_cols)
display(spearman_df)

In [0]:
# Convert correlation matrices to Pandas DataFrames, and display as heatmap
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Convert Spark DataFrames to Pandas DataFrames
pearson_pd = pearson_df.toPandas()
spearman_pd = spearman_df.toPandas()
pearson_pd.index = numeric_cols
spearman_pd.index = numeric_cols

# Plot Pearson correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(pearson_pd, annot=True, cmap="YlGnBu")
plt.title("Pearson Correlation Heatmap")
plt.show()

# Plot Spearman correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(spearman_pd, annot=True, cmap="YlGnBu")
plt.title("Spearman Correlation Heatmap")
plt.show()

### Descriptive statistics

In [0]:
# Examine distributions of each numeric column, except for male and churn, using DataBricks' built-in Visualization feature
display(df.select(numeric_cols))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Examine distribution of gender using DataBricks' built-in Visualization feature
display(df.select("Gender", "Churn"))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
display(df.select(numeric_cols).describe())

### Explore relationships visually

In [0]:
from pyspark.ml.feature import Bucketizer

# Bin Age for visual analysis
df = Bucketizer(
    splits=[18, 25, 35, 45, 55, 65, 75],
    inputCol="Age",
    outputCol="Age_Binned"
).transform(df)

df = df.withColumn("Age_Binned_Label", when(df.Age_Binned == 0, "18-25")
                   .when(df.Age_Binned == 1, "25-35")
                   .when(df.Age_Binned == 2, "35-45")
                   .when(df.Age_Binned == 3, "45-55")
                   .when(df.Age_Binned == 4, "55-65")
                   .when(df.Age_Binned == 5, "65-75"))

In [0]:
# Bin Annual_Income for visual analysis
df = Bucketizer(
    splits=[20000, 40000, 60000, 80000, 100000, 120000, 140000, 160000],
    inputCol="Annual_Income",
    outputCol="Annual_Income_Binned"
).transform(df)

df = df.withColumn("Annual_Income_Binned_Label", when(df.Annual_Income_Binned == 0, "1_$20K-$40K")
                   .when(df.Annual_Income_Binned == 1, "2_$40K-$60K")
                   .when(df.Annual_Income_Binned == 2, "3_$60K-$80K")
                   .when(df.Annual_Income_Binned == 3, "4_$80K-$100K")
                   .when(df.Annual_Income_Binned == 4, "5_$100K-$120K")
                   .when(df.Annual_Income_Binned == 5, "6_$120K-$140K")
                   .when(df.Annual_Income_Binned == 6, "7_$140K-$160K"))

In [0]:
# Bin Spending_Score for visual analysis
df = Bucketizer(
    splits=[0, 25, 50, 75, 100],
    inputCol="Spending_Score",
    outputCol="Spending_Score_Binned"
).transform(df)

df = df.withColumn("Spending_Score_Binned_Label", when(df.Spending_Score_Binned == 0, "0-25")
                   .when(df.Spending_Score_Binned == 1, "25-50")
                   .when(df.Spending_Score_Binned == 2, "50-75")
                   .when(df.Spending_Score_Binned == 3, "75-100"))

In [0]:
# Bin Membership_Years for visual analysis
df = Bucketizer(
    splits=[0, 2, 4, 6, 8, 10],
    inputCol="Membership_Years",
    outputCol="Membership_Years_Binned"
).transform(df)

df = df.withColumn("Membership_Years_Binned_Label", when(df.Membership_Years_Binned == 0, "0-2 years")
                   .when(df.Membership_Years_Binned == 1, "2-4 years")
                   .when(df.Membership_Years_Binned == 2, "4-6 years")
                   .when(df.Membership_Years_Binned ==3, "6-8 years")
                   .when(df.Membership_Years_Binned == 4, "8-10 years"))

In [0]:
# Bin Online_Purchases for visual analysis
df = Bucketizer(
    splits=[0, 25, 50, 75, 100, 125, 150, 175, 200],
    inputCol="Online_Purchases",
    outputCol="Online_Purchases_Binned"
).transform(df)

df = df.withColumn("Online_Purchases_Binned_Label", when(df.Online_Purchases_Binned == 0, "1_0-25")
                   .when(df.Online_Purchases_Binned == 1, "2_25-50")
                   .when(df.Online_Purchases_Binned == 2, "3_50-75")
                   .when(df.Online_Purchases_Binned == 3, "4_75-100")
                   .when(df.Online_Purchases_Binned == 4, "5_100-125")
                   .when(df.Online_Purchases_Binned == 5, "6_125-150")
                   .when(df.Online_Purchases_Binned == 6, "7_150-175")
                   .when(df.Online_Purchases_Binned == 7, "8_175-200"))

In [0]:
# Bin Discount_Usage for visual analysis
df = Bucketizer(
    splits=[0, 0.2, 0.4, 0.6, 0.8, 1],
    inputCol="Discount_Usage",
    outputCol="Discount_Usage_Binned"
).transform(df)

df = df.withColumn("Discount_Usage_Binned_Label", when(df.Discount_Usage_Binned == 0, "0-20%")
                   .when(df.Discount_Usage_Binned == 1, "20-40%")
                   .when(df.Discount_Usage_Binned == 2, "40-60%")
                   .when(df.Discount_Usage_Binned == 3, "60-80%")
                   .when(df.Discount_Usage_Binned == 4, "80-100%"))

In [0]:
# Inspect bivariate relationships between features and churn using DataBricks' built-in Visualization feature
X_feats = ["Age_Binned_Label", "Annual_Income_Binned_Label", "Spending_Score_Binned_Label", "Membership_Years_Binned_Label", "Online_Purchases_Binned_Label", "Discount_Usage_Binned_Label", "Gender"]

[display(df[X, "Churn"]) for X in X_feats]

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Inspect multi-variate relationships between features and churn using DataBricks' built-in Visualization feature
# Extract unique combinations of feature columns for heat map visualizations
from itertools import combinations
unique_combos = list(combinations(X_feats, 2))

# Display dataframes comprising unique combos and Churn column
for combo in unique_combos:
    combo = list(combo)
    combo.append('Churn')
    display(df[combo])

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

...

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.